# Week 10 Coding

**C1** checks filters against a specification. **C2** implements forward and backward filtering. **C3** (optional) compares the median filter with the average on a signal with outliers.

**Predict first.** Before running any check cell, write down on paper what the output will be.

**How to work in this notebook.** Execute the cells in order with Shift+Enter. Each task has a cell with gaps marked `...`: fill them, run the cell, then execute the check cell below it. A passing check prints a confirmation; a failing one raises an `AssertionError`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

## C1: Checking Filters against a Specification (P5, P6)

A specification is checked line by line, over the whole of each band, not only at its edges. The function `check` evaluates the magnitude response on a fine grid of frequencies from $0$ to $f_s/2$, with `signal.freqz(b, a, worN=f, fs=fs)`, which returns $H$ at the frequencies `f` in hertz.

A specification is a dictionary:
- `"passband"`: `(f_low, f_high, gain_min, gain_max)`, in hertz;
- `"stopbands"`: a list of `(f_low, f_high, gain_max)`, in hertz;
- `"delay"`: the largest admissible delay, in samples.

The delay of a filter is its largest delay in the passband, computed with `signal.group_delay`; for a filter with linear phase it is the constant delay $d$.

Complete `check`, which returns, for each line, the value found and whether the line is met.

In [ ]:
def check(b, a, fs, spec):
    """For each line of the specification: (value, met)."""
    f = np.linspace(0, fs / 2, 20001)                               # a fine grid of frequencies in hertz
    _, H = signal.freqz(b, a, worN=f, fs=fs)
    mag = np.abs(H)                                                 # |H| on the grid

    lo, hi, gmin, gmax = spec["passband"]
    inband = (f >= lo) & (f <= hi)                                  # True on the passband
    passband = ((mag[inband].min(), mag[inband].max()), gmin <= mag[inband].min() and mag[inband].max() <= gmax)

    stopbands = []
    for slo, shi, smax in spec["stopbands"]:
        largest = mag[(f >= slo) & (f <= shi)].max()                # the largest |H| on this stopband
        stopbands.append((largest, largest <= smax))

    _, gd = signal.group_delay((b, a), w=f[inband], fs=fs)          # the delay in samples on the passband (given)
    delay = (gd.max(), gd.max() <= spec["delay"] + 1e-9)
    return {"passband": passband, "stopbands": stopbands, "delay": delay}

**b) The filter of P5.** Before running the check cell, write down the values you computed by hand: the smallest $\lvert H\rvert$ in the passband, the largest $\lvert H\rvert$ in the stopband, and the delay.

In [ ]:
b5 = np.array([1, 2, 3, 2, 1]) / 9
spec5 = {"passband": (0, 50, 0.9, 1.1), "stopbands": [(300, 500, 0.1)], "delay": 5}   # 5 ms at 1000 Hz = 5 samples
r5 = check(b5, [1.0], 1000, spec5)
assert abs(r5["passband"][0][0] - 0.9358) < 1e-3 and r5["passband"][1], "P5 b): the passband is met, smallest |H| 0.9358"
assert abs(r5["stopbands"][0][0] - 1 / 9) < 1e-6 and not r5["stopbands"][0][1], "P5 c): the stopband is violated, 0.111 at 500 Hz"
assert abs(r5["delay"][0] - 2) < 1e-6 and r5["delay"][1], "P5 d): delay of 2 samples"
print("b) passed: passband met (0.936), stopband violated (0.111 at 500 Hz), delay 2 samples.")

**c) Three filters for the drone of P6.** The specification of P6: passband $0$ to $2$ Hz with $0.98 \le \lvert H\rvert \le 1.02$, stopband $80$ to $200$ Hz with $\lvert H\rvert \le 0.02$, delay at most $10$ samples, at $f_s = 400$ Hz. Three filters are proposed; how such filters are designed is the subject of Weeks 11 and 12. Complete `spec6`, check the three filters, and decide which one meets the specification.

In [ ]:
fs6 = 400
filters = {
    "filter 1": (np.ones(9) / 9, [1.0]),                            # the average of 9 samples
    "filter 2": (signal.firwin(31, 20, fs=fs6), [1.0]),             # 31 symmetric coefficients
    "filter 3": signal.butter(4, 20, fs=fs6),                       # a recursive filter of order 4
}
spec6 = {"passband": (0, 2, 0.98, 1.02), "stopbands": [(80, 200, 0.02)], "delay": 10}   # from P6
for name, (b, a) in filters.items():
    r = check(b, a, fs6, spec6)
    print(f"{name}: passband {r['passband'][1]}, stopband {r['stopbands'][0][1]} ({r['stopbands'][0][0]:.4f}), "
          f"delay {r['delay'][1]} ({r['delay'][0]:.1f} samples)")

In [ ]:
res = {name: check(b, a, fs6, spec6) for name, (b, a) in filters.items()}
meets = [name for name, r in res.items() if r["passband"][1] and r["stopbands"][0][1] and r["delay"][1]]
assert meets == ["filter 3"], "exactly one filter meets the specification"
assert not res["filter 1"]["stopbands"][0][1] and res["filter 1"]["delay"][1], "filter 1"
assert res["filter 2"]["stopbands"][0][1] and not res["filter 2"]["delay"][1], "filter 2"
print("c) passed: filter 3 meets the specification; filter 1 violates the stopband (0.146), filter 2 the delay (15 samples).")

**d)** Answer from the results:
1. Why does the average of 9 samples fail, although its delay is short?
2. Filter 2 has a linear phase. Why is this not enough here?
3. Plot $\lvert H\rvert$ of the three filters in decibels, with the bounds of the specification. Where does each of the failing filters leave the allowed region?

**Answers.**

1. Its magnitude has zeros at multiples of $400/9 = 44.4$ Hz, but between them it rises again: up to $0.146$ in the stopband, seven times the admissible $0.02$.
2. A linear phase preserves the shape, but its delay is $(31 - 1)/2 = 15$ samples, $37.5$ ms, more than the $10$ samples the controller tolerates. The specification requires a short delay, not a linear phase.
3. Filter 1 leaves the region in the stopband, between the zeros of its magnitude; filter 2 stays within the bounds of the magnitude and fails only the delay.

In [ ]:
f = np.linspace(0.5, fs6 / 2, 4000)
plt.figure(figsize=(8, 3.5))
for name, (b, a) in filters.items():
    _, H = signal.freqz(b, a, worN=f, fs=fs6)
    plt.semilogx(f, 20 * np.log10(np.abs(H) + 1e-12), label=name)
plt.fill_between([80, 200], 20 * np.log10(0.02), 10, color="red", alpha=0.1, label="not allowed in the stopband")
plt.fill_between([0.5, 2], -60, 20 * np.log10(0.98), color="red", alpha=0.1)
plt.ylim(-80, 5); plt.xlabel("frequency (Hz)"); plt.ylabel("|H| (dB)"); plt.legend(); plt.tight_layout()

## C2: Forward and Backward Filtering (P7, P9, P11)

A stored signal is filtered forward, the result is reversed in time, filtered again, and reversed again. `signal.lfilter(b, a, x)` applies the causal filter with coefficients `b` and `a`; `x[::-1]` reverses an array.

Complete `forward_backward`.

In [ ]:
def forward_backward(b, a, x):
    """Filter x forward and backward with the causal filter (b, a)."""
    y = signal.lfilter(b, a, x)                                     # the forward pass
    y = signal.lfilter(b, a, y[::-1])                               # the backward pass, on the reversed signal
    return y[::-1]                                                  # reverse again

**b) The filter of P7**, $h[0] = \tfrac23$, $h[1] = \tfrac13$. Before running the check cell, write down $g[-1]$, $g[0]$, $g[1]$ and the output of the two passes for the step at $n = -1, 0, 1$.

In [ ]:
h7 = np.array([2, 1]) / 3
x = np.zeros(41); x[20] = 1                                         # an impulse at the centre
g = forward_backward(h7, [1.0], x)
assert np.allclose(g[19:22], [2 / 9, 5 / 9, 2 / 9]) and np.allclose(np.delete(g, [19, 20, 21]), 0), "P7 b)"
step = np.r_[np.zeros(20), np.ones(30)]                             # a step at index 20
y1, y2 = signal.lfilter(h7, [1.0], step), forward_backward(h7, [1.0], step)
assert np.allclose(y1[19:22], [0, 2 / 3, 1]) and np.allclose(y2[18:22], [0, 2 / 9, 7 / 9, 1]), "P7 a) and d)"
print("b) passed: g = {2/9, 5/9, 2/9}, even; the step rises at 2/9, 7/9, 1, symmetric about the step.")

**c) The filter of P9**, $y[n] = 0.5\,y[n-1] + 0.5\,u[n]$, that is, `b = [0.5]`, `a = [1, -0.5]`. The input of P9 c) is filtered once and forward and backward. The check cell compares both outputs with the formulas of P9, away from the two ends of the signal, where the passes start.

In [ ]:
b9, a9 = [0.5], [1, -0.5]
n = np.arange(400)
u = np.cos(np.pi * n / 4) + np.cos(np.pi * n / 2)
y_once, y_fb = signal.lfilter(b9, a9, u), forward_backward(b9, a9, u)
inner = slice(100, 300)
pred_once = 0.6786 * np.cos(np.pi / 4 * (n - 0.6372)) + 0.4472 * np.cos(np.pi / 2 * (n - 0.2952))
pred_fb = 0.4605 * np.cos(np.pi * n / 4) + 0.2 * np.cos(np.pi * n / 2)
assert np.allclose(y_once[inner], pred_once[inner], atol=1e-3), "P9 c)"
assert np.allclose(y_fb[inner], pred_fb[inner], atol=1e-3), "P9 d)"
print("c) passed: once, the components are delayed by 0.64 and 0.30 samples; forward and backward, by none.")

s = np.r_[np.zeros(40), np.ones(40)]
plt.figure(figsize=(8, 3))
plt.plot(s, "k:", label="step")
plt.plot(signal.lfilter(b9, a9, s), ".-", label="once (causal)")
plt.plot(forward_backward(b9, a9, s), ".-", label="forward and backward")
plt.xlim(30, 55); plt.xlabel("n"); plt.legend(); plt.tight_layout()

**d)** In the plot of the step: which output lags the step, and which is symmetric about it? Which output uses samples after $n$?

**Answer.** The causal output starts to rise at the step and approaches $1$ afterwards: it lags. The forward and backward output starts to rise before the step and is symmetric about it; it uses the samples after $n$, which only a stored signal provides.

## C3 (optional): The Median Filter against Outliers (P8, P12)

Complete `median3`, the median of $u[n-1]$, $u[n]$, $u[n+1]$; the first and the last sample are kept unchanged. `np.median(..., axis=0)` takes the median of the rows of a stacked array.

In [ ]:
def median3(x):
    """The median of three samples; the two end samples are kept."""
    y = np.array(x, dtype=float)
    y[1:-1] = np.median(np.stack([x[:-2], x[1:-1], x[2:]]), axis=0) # median of x[n-1], x[n], x[n+1]
    return y

In [ ]:
n = np.arange(-1, 7)
y8 = median3(np.cos(np.pi * n / 3))[1:7]                            # n = 0..5
assert np.allclose(y8, [0.5, 0.5, -0.5, -0.5, -0.5, 0.5]), "P8 a)"
assert np.allclose(np.fft.fft(y8), [0, 2, 0, -1, 0, 2]), "P8 c): a component at Omega = pi (row 3)"
u1, u2 = np.array([0, 2, 0.]), np.array([0, -1, 1.])
assert median3(u1)[1] == 0 and median3(u2)[1] == 0 and median3(u1 + u2)[1] == 1, "P12 c)"
print("P8 and P12 passed: the median creates the frequency pi and is not additive.")

**A slow signal with outliers.** A sinusoid with $100$ samples per period contains five isolated outliers. Before running the cell, predict which filter leaves the larger error near an outlier.

In [ ]:
n = np.arange(500)
clean = np.sin(2 * np.pi * n / 100)
x = clean.copy(); x[[60, 170, 245, 330, 410]] += 3.0                # five isolated outliers
y_med = median3(x)
y_avg = np.convolve(x, np.ones(3) / 3, mode="same")
err_med, err_avg = np.abs(y_med - clean)[1:-1].max(), np.abs(y_avg - clean)[1:-1].max()
print(f"largest error: median {err_med:.3f}, average {err_avg:.3f}")
assert err_med < 0.07 and err_avg > 0.99
plt.figure(figsize=(8, 3))
plt.plot(x, color="0.7", label="with outliers"); plt.plot(y_avg, label="average of 3"); plt.plot(y_med, label="median of 3")
plt.xlim(150, 260); plt.xlabel("n"); plt.legend(); plt.tight_layout()

**Questions.**
1. Why does the average leave an error of $1$ at three samples around each outlier?
2. Where does the median change the clean signal, and by how much at most?

**Answers.**

1. Each of the three averages that contain the outlier adds $3/3 = 1$: the outlier is reduced to a third and spread over three samples.
2. Only where the clean signal has a peak or a trough, where the middle sample is not the median of its neighbourhood: the median replaces it by a neighbour, an error of at most the change between two samples, below $0.07$ here. Away from the extremes the clean signal is monotonic over three samples and the median leaves it unchanged.